# 预录语音 → 音响；麦克风同时闭环采样

只开这一本音频Notebook。录音2秒；随后直放/旁路/抵消各约2.4秒，总发声最多7.2秒，含上传配置整轮约5–10分钟。箱盖、接线和两只音响位置不变。录音时音响断电；播放测试时手机停止外放。持续尖叫立即切断音响电源。

## 第1格：核验文件（不发声）
点击下面代码格，Shift+Enter一次；看到 REPLAY_PACKAGE_OK 才继续。

In [ ]:
from pathlib import Path
import sys, os, importlib
BOARD = Path('/home/xilinx/jupyter_notebooks/base/audio/audio_replay')
if not BOARD.is_dir():
    raise FileNotFoundError('请先把完整程序包上传到 ' + str(BOARD))
os.chdir(str(BOARD))
if str(BOARD) not in sys.path: sys.path.insert(0, str(BOARD))
import replay_tools, replay_session
importlib.reload(replay_tools)
importlib.reload(replay_session)
if 'replay_trial' in sys.modules: importlib.reload(sys.modules['replay_trial'])
replay_session.check_package()
replay_session.load_scene(BOARD.parent / 'audio_closed/field_setup.json')
print('SCENE_FILE_OK; NO_AUDIO_STARTED')

## 第2格：录一次语音（音响断电）
先运行本格。等 RECORD 提示出现后，让手机在原位置播放语音，输入 RECORD 回车，立即录2秒。录完停止手机外放。source.wav已存在则复用，不重复录音。
录音只保存在板卡；已有0.5–2.5秒PCM WAV也可命名source.wav放本目录并跳过录音。

In [ ]:
source_path = replay_session.record_source()

## 第3格：手机静音，连续完成三项
先停止手机外放，恢复音响供电，保持箱体/接线位置不变。运行本格，现场就绪后输入 READY。每项配置完成后，只在准备听时输入 START，该项立即播放并采样约2.4秒，再自动静音。

结束后输入：0无声、1清楚语音且结束安静、2杂音、3尖叫增长、4结束仍响。只有1继续，其他值停止。本轮不升增益。持续响声先断音响电源。

结束后只下载最后显示路径的 summary.json。原始WAV/NPZ留板。

In [ ]:
result = replay_session.run_replay()